In [2]:
import pandas as pd 
import numpy as np

In [3]:
df=pd.read_csv("housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [4]:
df["income_cat"]=pd.cut(df["median_income"],bins=[0,1.5,2.5,3.5,4.5,5.5,6,np.inf],labels=[1,2,3,4,5,6,7])

In [5]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,income_cat
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,7
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,7
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,7
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,6
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,4


In [6]:
from sklearn.model_selection import StratifiedShuffleSplit # can do multiple splits

In [7]:
sss=StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=42)
for train_idx,test_idx in sss.split(df,df["income_cat"]):
    train_set=df.iloc[train_idx]
    test_set=df.iloc[test_idx]

In [8]:
#droping "income_cat" column now

In [9]:
df=train_set.copy()

In [10]:
df.drop("income_cat",axis=1,inplace=True)

In [12]:
housing=df.drop("median_house_value",axis=1)

In [14]:
housing_label=df["median_house_value"].copy()

In [19]:
from sklearn.impute import SimpleImputer
imputer=SimpleImputer(strategy="median")
housing_num=housing.select_dtypes(include="number")
imputer.fit(housing_num)

,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False


In [20]:
#Inspection
imputer.statistics_

array([-118.51  ,   34.26  ,   29.    , 2125.    ,  433.    , 1165.    ,
        409.    ,    3.5346])

In [22]:
X=imputer.transform(housing_num)

In [23]:
X

array([[-121.84  ,   37.33  ,   26.    , ..., 2059.    ,  416.    ,
           3.6765],
       [-121.49  ,   38.51  ,   30.    , ..., 1857.    ,  579.    ,
           3.1768],
       [-120.25  ,   38.55  ,   15.    , ..., 1103.    ,  433.    ,
           3.0125],
       ...,
       [-116.41  ,   33.74  ,   17.    , ...,  958.    ,  440.    ,
           2.4659],
       [-117.94  ,   34.1   ,   31.    , ...,  929.    ,  244.    ,
           3.3625],
       [-122.47  ,   38.3   ,   15.    , ..., 2175.    ,  924.    ,
           3.4031]], shape=(16512, 8))

In [24]:
# To convert these to Data Frame back:
housing_X=pd.DataFrame(X,columns=housing_num.columns,index=housing.index)

In [25]:
housing_X.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
17751,-121.84,37.33,26.0,1934.0,408.0,2059.0,416.0,3.6765
12609,-121.49,38.51,30.0,3166.0,607.0,1857.0,579.0,3.1768
1030,-120.25,38.55,15.0,4403.0,891.0,1103.0,433.0,3.0125
9844,-121.90,36.59,42.0,2689.0,510.0,1023.0,459.0,4.6182
1253,-122.01,39.21,50.0,1592.0,372.0,781.0,307.0,2.2679


In [28]:
df.isnull().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        162
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

In [29]:
housing_X.isnull().sum() #here you can see missing value of total bedrooms are handled

longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
dtype: int64